In [ ]:
from resources.imports import *

from resources.lattices import Geometry, calcC_mohr, calcC_sims, plot_IsotropyVariation, calc_IsoEffProperties, effProperties, \
                                prismatic_plane_strain

In [ ]:
LAT = "FCC"
dis = 'disNodes'

# TOOD: Check homogenization sims for all lattice topologies.

# Truss Axial Plasticity Method

In [ ]:
l = 10
nnx = 10

geom = Geometry(LAT, l, nnx)

In [ ]:
Cs_mohr = calcC_mohr(copy.deepcopy(geom), "lattice", dis=dis, count=20, plot=False)
C_per = Cs_mohr[0]
Cunit_mohr = calcC_mohr(copy.deepcopy(geom), "unit", plot=False)[0]

In [ ]:
plot_IsotropyVariation(Cs_mohr, stiff=True, paper=True)

In [ ]:
E_eff, v_eff, iso = calc_IsoEffProperties(Cunit_mohr)
E_eff, v_eff, iso, 123e9

In [ ]:
effProperties(LAT, geom)

In [ ]:
Cunit_mohr

In [ ]:
# Cunit_mohr uses physical constituent inputs and specimen x/y axes.
# Reduce once using E_L,3 = rho_bar*E_s and nu_L,31=nu_L,32=nu_s.
# A measured tensor from already-modified beams must instead declare
# C_state="plane_strain"; that bypasses the mixture/reduction.
elastic_pe = prismatic_plane_strain(Cunit_mohr, geom.rD, E_s=123e9, v_s=.3,
                                    C_state="plane_stress", crack_axis=0)
lambda_L_pe = elastic_pe["lambda_pe"]  # axis1=x along crack, axis2=y opening
chi_pe = elastic_pe["chi_pe"]
lambda_O_pe = 1/lambda_L_pe  # Ortega text puts axis1 along opening
M_I_pe = elastic_pe["M_I_pe"]
# Canonical FCC has lambda=1. Unequal axial moduli require independent
# verification of the C(T) fit's source-axis mapping before use.
{"E_L3_Pa": elastic_pe["E3"], "lambda_L_pe": lambda_L_pe,
 "lambda_O_pe": lambda_O_pe, "chi_pe": chi_pe, "M_I_pe_Pa": M_I_pe}


In [ ]:
v_eff = 1/3
C_iso_pStress = (1/(1-v_eff**2))*np.array([[1, v_eff, 0],
                                           [v_eff, 1, 0],
                                           [0, 0, (1-v_eff)/2]])
C_iso_pStrain = (1/((1-2*v_eff)*(1+v_eff)))*np.array([[1-v_eff, v_eff, 0],
                                                      [v_eff, 1-v_eff, 0],
                                                      [0, 0, (1-2*v_eff)/2]])

np.around(C_iso_pStress/min(C_iso_pStress[0][:2]),2), np.around(C_iso_pStrain/min(C_iso_pStrain[0][:2]))

# Beam-Based Homogenization Method

In [ ]:
nnx = 20
if LAT.lower() == "tri":
    nnx = 30

Cs_sims = calcC_sims(LAT, nnx, dis=dis, count=0, pDir=r"Z:\p1\sims\Ti\stiffMatrix\Cmatrix-periodic\\")

In [ ]:
plot_IsotropyVariation(Cs_sims, paper=True)